# Article statistics

Inspect downloaded article metadata stored in `data/articles.db`.

In [ ]:
from pathlib import Path

import polars as pl

In [ ]:
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
else:
    PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "data" / "articles.db"
DB_URI = f"sqlite:///{DB_PATH.expanduser().resolve().as_posix()}"
print(f"Reading articles from {DB_URI}")

Reading articles from sqlite:////Users/matthiaskaeding/Projects/econlit/data/articles.db


In [ ]:
QUERY = "SELECT * FROM articles"
df = pl.read_database_uri(QUERY, DB_URI)
print(f"Loaded {len(df)} articles")

Loaded 5746 articles


## Overview

In [ ]:
summary = pl.DataFrame(
    {
        "metric": ["Total articles", "DataFrame shape", "Columns"],
        "value": [
            str(len(df)),
            f"{df.shape[0]} rows x {df.shape[1]} cols",
            ", ".join(df.columns),
        ],
    }
)
print("\n" + "=" * 60)
print("ARTICLES DATABASE OVERVIEW")
print("=" * 60)
print(summary)


ARTICLES DATABASE OVERVIEW
shape: (3, 2)
┌─────────────────┬─────────────────────────────────┐
│ metric          ┆ value                           │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ Total articles  ┆ 5746                            │
│ DataFrame shape ┆ 5746 rows x 9 cols              │
│ Columns         ┆ id, doi, title, abstract, jour… │
└─────────────────┴─────────────────────────────────┘


## Sample data

In [ ]:
print("\n" + "-" * 60)
print("SAMPLE DATA (first 5 rows)")
print("-" * 60)
sample_rows = df.head()
print(sample_rows)

## Articles by journal

In [ ]:
print("\n" + "-" * 60)
print("ARTICLES BY JOURNAL")
print("-" * 60)
by_journal = df.group_by("journal").len().sort("len", descending=True)
print(by_journal)


------------------------------------------------------------
ARTICLES BY JOURNAL
------------------------------------------------------------
shape: (5, 2)
┌────────────────────────────────┬──────┐
│ journal                        ┆ len  │
│ ---                            ┆ ---  │
│ str                            ┆ u32  │
╞════════════════════════════════╪══════╡
│ American Economic Review       ┆ 1886 │
│ Journal of Political Economy   ┆ 1242 │
│ Econometrica                   ┆ 1096 │
│ Review of Economic Studies     ┆ 969  │
│ Quarterly Journal of Economics ┆ 553  │
└────────────────────────────────┴──────┘


## Articles by year

In [ ]:
print("\n" + "-" * 60)
print("ARTICLES BY YEAR")
print("-" * 60)
by_year = df.group_by("year").len().sort("year")
print(by_year)


------------------------------------------------------------
ARTICLES BY YEAR
------------------------------------------------------------
shape: (11, 2)
┌──────┬─────┐
│ year ┆ len │
│ ---  ┆ --- │
│ i64  ┆ u32 │
╞══════╪═════╡
│ 2015 ┆ 536 │
│ 2016 ┆ 561 │
│ 2017 ┆ 605 │
│ 2018 ┆ 420 │
│ 2019 ┆ 436 │
│ …    ┆ …   │
│ 2021 ┆ 544 │
│ 2022 ┆ 502 │
│ 2023 ┆ 510 │
│ 2024 ┆ 511 │
│ 2025 ┆ 571 │
└──────┴─────┘


## Articles by source

In [ ]:
print("\n" + "-" * 60)
print("ARTICLES BY SOURCE")
print("-" * 60)
by_source = df.group_by("source").len().sort("len", descending=True)
print(by_source)


------------------------------------------------------------
ARTICLES BY SOURCE
------------------------------------------------------------
shape: (2, 2)
┌──────────┬──────┐
│ source   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ crossref ┆ 5446 │
│ openalex ┆ 300  │
└──────────┴──────┘


## Abstract coverage

In [ ]:
has_abstract = df.filter(pl.col("abstract").is_not_null()).height
abstract_summary = pl.DataFrame(
    {
        "status": ["With abstract", "Without abstract"],
        "count": [has_abstract, len(df) - has_abstract],
    }
).with_columns(pl.col("count").truediv(len(df)).mul(100).alias("percent"))
print("\n" + "-" * 60)
print("ABSTRACT COVERAGE")
print("-" * 60)
print(abstract_summary)


------------------------------------------------------------
ABSTRACT COVERAGE
------------------------------------------------------------
shape: (2, 3)
┌──────────────────┬───────┬───────────┐
│ status           ┆ count ┆ percent   │
│ ---              ┆ ---   ┆ ---       │
│ str              ┆ i64   ┆ f64       │
╞══════════════════╪═══════╪═══════════╡
│ With abstract    ┆ 3295  ┆ 57.344239 │
│ Without abstract ┆ 2451  ┆ 42.655761 │
└──────────────────┴───────┴───────────┘


### Abstract coverage by journal

In [ ]:
print("\n" + "-" * 60)
print("ABSTRACT COVERAGE BY JOURNAL")
print("-" * 60)
abstract_by_journal = df.group_by("journal").agg(
    [
        pl.len().alias("total"),
        pl.col("abstract").is_not_null().sum().alias("with_abstract"),
    ]
)
abstract_by_journal = abstract_by_journal.with_columns(
    (pl.col("total") - pl.col("with_abstract")).alias("without_abstract")
).with_columns(
    (pl.col("without_abstract") / pl.col("total") * 100).alias("percent_without")
)
abstract_by_journal = abstract_by_journal.select(
    [
        pl.col("journal"),
        pl.col("without_abstract"),
        pl.col("percent_without"),
    ]
).sort("percent_without", descending=True)
print(abstract_by_journal)


------------------------------------------------------------
ABSTRACT COVERAGE BY JOURNAL
------------------------------------------------------------
shape: (5, 3)
┌────────────────────────────────┬──────────────────┬─────────────────┐
│ journal                        ┆ without_abstract ┆ percent_without │
│ ---                            ┆ ---              ┆ ---             │
│ str                            ┆ u32              ┆ f64             │
╞════════════════════════════════╪══════════════════╪═════════════════╡
│ Journal of Political Economy   ┆ 1236             ┆ 99.516908       │
│ Econometrica                   ┆ 599              ┆ 54.653285       │
│ Review of Economic Studies     ┆ 275              ┆ 28.379773       │
│ Quarterly Journal of Economics ┆ 97               ┆ 17.540687       │
│ American Economic Review       ┆ 244              ┆ 12.937434       │
└────────────────────────────────┴──────────────────┴─────────────────┘


### Abstract coverage by source

In [ ]:
print("\n" + "-" * 60)
print("ABSTRACT COVERAGE BY SOURCE")
print("-" * 60)
abstract_by_source = df.group_by("source").agg(
    [
        pl.len().alias("total"),
        pl.col("abstract").is_not_null().sum().alias("with_abstract"),
    ]
)
abstract_by_source = abstract_by_source.with_columns(
    (pl.col("total") - pl.col("with_abstract")).alias("without_abstract")
).with_columns(
    (pl.col("without_abstract") / pl.col("total") * 100).alias("percent_without")
)
abstract_by_source = abstract_by_source.select(
    [
        pl.col("source"),
        pl.col("without_abstract"),
        pl.col("percent_without"),
    ]
).sort("percent_without", descending=True)
print(abstract_by_source)


------------------------------------------------------------
ABSTRACT COVERAGE BY SOURCE
------------------------------------------------------------
shape: (2, 3)
┌──────────┬──────────────────┬─────────────────┐
│ source   ┆ without_abstract ┆ percent_without │
│ ---      ┆ ---              ┆ ---             │
│ str      ┆ u32              ┆ f64             │
╞══════════╪══════════════════╪═════════════════╡
│ crossref ┆ 2393             ┆ 43.940507       │
│ openalex ┆ 58               ┆ 19.333333       │
└──────────┴──────────────────┴─────────────────┘
